# Data Cleaning and Pre-Processing

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
folder = "../data/raw/OBD-II-Dataset/"
files = sorted(os.listdir(folder))

print("Total files:", len(files))

In [ ]:
def normalize_columns(df):

    df.columns = (
        df.columns
        .str.strip()
        .str.replace("Ã‚Â", "", regex=False)
        .str.replace("Â", "", regex=False)
    )

    return df

In [ ]:
dfs = []

for i, file in enumerate(files):

    if file.endswith(".csv"):

        df = pd.read_csv(folder + file)

        df = normalize_columns(df)

        df["trip_id"] = i + 1

        dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Rows:", len(data))
print("Trips:", data["trip_id"].nunique())

In [ ]:
data = data.rename(columns={
    "Vehicle Speed Sensor [km/h]": "speed",
    "Engine RPM [RPM]": "rpm",
    "Absolute Throttle Position [%]": "throttle",
    "Intake Manifold Absolute Pressure [kPa]": "map",
    "Air Flow Rate from Mass Flow Sensor [g/s]": "maf",
    "Accelerator Pedal Position D [%]": "pedal_d",
    "Accelerator Pedal Position E [%]": "pedal_e",
    "Engine Coolant Temperature [°C]": "coolant_temp",
    "Intake Air Temperature [°C]": "intake_temp",
    "Ambient Air Temperature [°C]": "ambient_temp"
})

In [ ]:
def convert_time(t):

    parts = str(t).split(":")

    if len(parts) == 3:
        h, m, s = parts
        return int(h)*3600 + int(m)*60 + float(s)

    elif len(parts) == 2:
        m, s = parts
        return int(m)*60 + float(s)

    else:
        return np.nan

data["time_sec"] = data["Time"].apply(convert_time)

In [ ]:
data = data.dropna(subset=["time_sec"])

In [ ]:
data = data.sort_values(["trip_id","time_sec"])

In [ ]:
data["dt"] = data.groupby("trip_id")["time_sec"].diff()

In [ ]:
data["dt"].describe()

In [ ]:
data = data[(data["dt"] > 0) & (data["dt"] < 1)]

In [ ]:
data["time_clean"] = data.groupby("trip_id")["time_sec"].transform(lambda x: x - x.min())

In [ ]:
data = data.sort_values(["trip_id", "time_sec"])

cols = data.columns.drop("trip_id")

data[cols] = data.groupby("trip_id")[cols].ffill().bfill()

In [ ]:
data = data.reset_index(drop=True)

In [ ]:
data = data.rename(columns={"time_clean": "time"})

data = data.drop(columns=["Time", "time_sec", "pedal_e"], errors="ignore")

In [ ]:
data.isnull().sum()

In [ ]:
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])
print("Trips:", data["trip_id"].nunique())

In [ ]:
data = data[
    ["trip_id", "time"] +
    [col for col in data.columns if col not in ["trip_id", "time"]]
]

In [ ]:
print(data.columns)

In [ ]:
data.to_csv("../data/processed/clean_obd_data.csv", index=False)

SAMPLE

In [ ]:
import numpy as np

sample_fraction = 0.25

samples = []

for trip_id, trip in data.groupby("trip_id"):

    trip = trip.sort_values("time")

    n = len(trip)
    sample_size = int(n * sample_fraction)

    start = np.random.randint(0, n - sample_size)

    samples.append(trip.iloc[start:start + sample_size])

sample_data = pd.concat(samples)

print("Sample rows:", len(sample_data))
print("Trips:", sample_data["trip_id"].nunique())

In [ ]:
sample_data.to_csv("../data/processed/clean_obd_sample.csv", index=False)

In [ ]:
data.info(memory_usage="deep")

In [ ]:
for col in data.select_dtypes(include=["float64"]).columns:
    data[col] = pd.to_numeric(data[col], downcast="float")

for col in data.select_dtypes(include=["int64"]).columns:
    data[col] = pd.to_numeric(data[col], downcast="integer")

In [ ]:
data.info(memory_usage="deep")